# Week 16 Optional: AgentCore Runtime Deployment

## Overview

This optional notebook covers advanced AgentCore topics for students who want
to go deeper into production agent deployment. It is **NOT required** for the course.

## What You'll Learn

1. **AgentCore Runtime** — Package and deploy a Strands agent to managed containers
2. **AgentCore Gateway** — Convert existing APIs into MCP-compatible tools
3. **Human-in-the-Loop** — Add approval gates for high-risk decisions
4. **Advanced Multi-Agent Patterns** — Supervisor vs Arbiter, parallel execution

## Prerequisites

- Completed Week 16 main notebook (Strands agents, multi-agent, AgentCore Memory)
- AWS credentials with AgentCore permissions

In [ ]:
# =============================================================================
# SETUP
# =============================================================================

!pip install -q strands-agents strands-agents-tools bedrock-agentcore bedrock-agentcore-starter-toolkit

import os
import json
import boto3
import sagemaker
from sagemaker import get_execution_role
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore import BedrockAgentCoreApp

# SageMaker + Bedrock connection (same as main notebook)
sess = sagemaker.Session()
role = get_execution_role()
AWS_REGION = sess.boto_region_name

print(f"SageMaker execution role: {role.split('/')[-1]}")
print(f"AWS Region: {AWS_REGION}")

# Verify Bedrock
bedrock_client = boto3.client('bedrock', region_name=AWS_REGION)
try:
    models = bedrock_client.list_foundation_models()
    print(f"Connected to Bedrock! {len(models['modelSummaries'])} models available.")
except Exception as e:
    print(f"Bedrock connection failed: {e}")

# Section 1: AgentCore Runtime Deployment

## What is AgentCore Runtime?

AgentCore Runtime hosts your agent code in **managed containers**. You package
your Strands agent as a Python application, deploy it, and AWS handles:
- Container orchestration and scaling
- Load balancing across requests
- Health checks and auto-recovery
- Secure networking and IAM integration

## The Deployment Pattern

```python
from bedrock_agentcore import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel

# 1. Create your app
app = BedrockAgentCoreApp()

# 2. Define your agent as the entrypoint
@app.entrypoint
def handle_request(user_input: str) -> str:
    agent = Agent(
        model=BedrockModel(model_id="us.anthropic.claude-3-haiku-20240307-v1:0"),
        tools=[my_tool_1, my_tool_2],
        system_prompt="You are a fraud investigator."
    )
    result = agent(user_input)
    return str(result)
```

Then deploy with the CLI:
```bash
agentcore configure -e my_agent.py
agentcore launch
```

After deployment, invoke via boto3:
```python
client = boto3.client('bedrock-agentcore')
response = client.invoke_agent_runtime(
    agentRuntimeArn="arn:aws:bedrock-agentcore:us-east-1:123456:runtime/my-agent",
    input={"text": "Investigate TXN-001"}
)
```

In [ ]:
# =============================================================================
# DEMO: Package a Fraud Agent for AgentCore Runtime
# =============================================================================
# We'll create the agent entrypoint file that AgentCore Runtime will execute.
# Note: actual deployment requires CLI commands — we'll write the file here
# and show the deploy commands.

# Re-create the fraud tools (same as main notebook)
@tool
def rt_lookup_transaction(transaction_id: str) -> str:
    """Look up transaction details by ID.

    Args:
        transaction_id: The transaction ID to look up
    """
    transactions = {
        "TXN-001": {"amount": 4500, "type": "wire_transfer", "merchant": "Unknown Overseas",
                     "time": "3:47 AM", "location": "Lagos, Nigeria"},
        "TXN-002": {"amount": 89.99, "type": "subscription", "merchant": "Netflix",
                     "time": "6:00 PM", "location": "Chicago, IL"},
    }
    txn = transactions.get(transaction_id)
    return json.dumps(txn, indent=2) if txn else f"{transaction_id} not found."

@tool
def rt_calculate_risk(amount: float, is_international: bool) -> str:
    """Calculate a basic risk score.

    Args:
        amount: Transaction amount in dollars
        is_international: Whether the transaction crosses borders
    """
    score = 0
    if amount > 1000: score += 40
    if is_international: score += 40
    level = "LOW" if score < 30 else "MEDIUM" if score < 60 else "HIGH"
    return json.dumps({"risk_score": score, "risk_level": level})

# Write the entrypoint file
entrypoint_code = '''
from bedrock_agentcore import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
import json

app = BedrockAgentCoreApp()

@tool
def lookup_transaction(transaction_id: str) -> str:
    """Look up transaction details by ID.
    Args:
        transaction_id: The transaction ID to look up
    """
    # In production, this would query a real database
    transactions = {
        "TXN-001": {"amount": 4500, "type": "wire_transfer", "merchant": "Unknown Overseas"},
    }
    txn = transactions.get(transaction_id)
    return json.dumps(txn) if txn else f"{transaction_id} not found."

@app.entrypoint
def handle_request(user_input: str) -> str:
    agent = Agent(
        model=BedrockModel(model_id="us.anthropic.claude-3-haiku-20240307-v1:0"),
        tools=[lookup_transaction],
        system_prompt="You are a fraud investigator. Investigate transactions and provide verdicts."
    )
    result = agent(user_input)
    return str(result)
'''

with open('fraud_agent_runtime.py', 'w') as f:
    f.write(entrypoint_code)

print("Created fraud_agent_runtime.py")
print()
print("To deploy to AgentCore Runtime, run in terminal:")
print("  agentcore configure -e fraud_agent_runtime.py")
print("  agentcore launch")
print()
print("After deployment, invoke via boto3:")
print("  client = boto3.client('bedrock-agentcore')")
print("  response = client.invoke_agent_runtime(")
print("      agentRuntimeArn='<your-arn>',")
print("      input={'text': 'Investigate TXN-001'})")

## Lab 1: Write Your Own AgentCore Entrypoint (20 minutes)

### Your Task

Create a production-ready agent entrypoint that handles the full multi-agent
fraud pipeline from the main notebook (Triage + Investigation + Decision).

### Steps

1. **Write a `multi_agent_runtime.py`** file that:
   - Defines all 4 fraud tools (`lookup_transaction`, `check_customer_history`,
     `calculate_risk_score`, `check_fraud_policy`)
   - Creates 3 specialist agents (triage, investigation, decision)
   - Wraps each as `@tool` for the supervisor
   - Uses `@app.entrypoint` to expose the supervisor agent
2. **Add error handling** — if any sub-agent fails, catch the exception and
   return a structured error response instead of crashing
3. **Test locally** by calling `handle_request()` directly before deploying

### Hints

- Use `try/except` inside each `run_*` wrapper tool
- Return `json.dumps({"error": str(e)})` on failure
- The `@app.entrypoint` function should accept a string and return a string

In [ ]:
# =============================================================================
# SOLUTION: LAB 1 — MULTI-AGENT AGENTCORE ENTRYPOINT
# =============================================================================

multi_agent_code = '''
from bedrock_agentcore import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
import json

app = BedrockAgentCoreApp()
MODEL_ID = "us.anthropic.claude-3-haiku-20240307-v1:0"
llm = BedrockModel(model_id=MODEL_ID)

# --- Fraud tools ---
@tool
def lookup_transaction(transaction_id: str) -> str:
    """Look up transaction details by ID.
    Args:
        transaction_id: The transaction ID to look up
    """
    transactions = {
        "TXN-001": {"amount": 4500, "type": "wire_transfer", "merchant": "Unknown Overseas",
                     "time": "3:47 AM", "location": "Lagos, Nigeria"},
        "TXN-002": {"amount": 89.99, "type": "subscription", "merchant": "Netflix",
                     "time": "6:00 PM", "location": "Chicago, IL"},
    }
    txn = transactions.get(transaction_id)
    return json.dumps(txn) if txn else f"{transaction_id} not found."

@tool
def check_customer_history(customer_id: str) -> str:
    """Retrieve customer history.
    Args:
        customer_id: The customer ID to look up
    """
    histories = {
        "CUST-001": {"avg_transaction": 250, "international_transfers": 0, "account_age_years": 3},
    }
    hist = histories.get(customer_id)
    return json.dumps(hist) if hist else f"{customer_id} not found."

@tool
def calculate_risk_score(amount: float, is_international: bool,
                         is_unusual_hour: bool, merchant_known: bool) -> str:
    """Calculate fraud risk score.
    Args:
        amount: Transaction amount
        is_international: Whether international
        is_unusual_hour: Whether unusual hour
        merchant_known: Whether merchant is known
    """
    score = 0
    if amount > 1000: score += 30
    if is_international: score += 25
    if is_unusual_hour: score += 20
    if not merchant_known: score += 25
    level = "LOW" if score < 30 else "MEDIUM" if score < 60 else "HIGH"
    return json.dumps({"risk_score": score, "risk_level": level})

@tool
def check_fraud_policy(risk_level: str, amount: float) -> str:
    """Check fraud policy.
    Args:
        risk_level: LOW, MEDIUM, or HIGH
        amount: Transaction amount
    """
    policies = {"LOW": "Auto-approve.", "MEDIUM": "Flag for review.", "HIGH": "Block immediately."}
    return json.dumps({"policy": policies.get(risk_level, "Unknown"), "auto_escalate": amount > 5000})

# --- Specialist agents ---
triage = Agent(model=llm, tools=[lookup_transaction, calculate_risk_score],
               system_prompt="You are a triage specialist. Classify risk quickly.", callback_handler=None)
investigator = Agent(model=llm, tools=[lookup_transaction, check_customer_history],
                     system_prompt="You are an investigator. Gather evidence thoroughly.", callback_handler=None)
decider = Agent(model=llm, tools=[check_fraud_policy],
                system_prompt="You are a decision maker. Render final verdict.", callback_handler=None)

# --- Wrap as tools with error handling ---
@tool
def run_triage(transaction_id: str, customer_id: str) -> str:
    """Triage a transaction.
    Args:
        transaction_id: Transaction ID
        customer_id: Customer ID
    """
    try:
        return str(triage(f"Triage {transaction_id} for {customer_id}."))
    except Exception as e:
        return json.dumps({"error": str(e), "fallback": "Route to human review"})

@tool
def run_investigation(transaction_id: str, customer_id: str, triage_result: str) -> str:
    """Investigate a transaction.
    Args:
        transaction_id: Transaction ID
        customer_id: Customer ID
        triage_result: Triage result
    """
    try:
        return str(investigator(f"Investigate {transaction_id} for {customer_id}. Triage: {triage_result}"))
    except Exception as e:
        return json.dumps({"error": str(e), "fallback": "Route to human review"})

@tool
def run_decision(triage_result: str, investigation_report: str, amount: float) -> str:
    """Render verdict.
    Args:
        triage_result: Triage result
        investigation_report: Investigation report
        amount: Transaction amount
    """
    try:
        return str(decider(f"Verdict. Triage: {triage_result}. Investigation: {investigation_report}. Amount: ${amount}"))
    except Exception as e:
        return json.dumps({"error": str(e), "fallback": "Route to human review"})

# --- Supervisor entrypoint ---
supervisor = Agent(model=llm, tools=[run_triage, run_investigation, run_decision],
                   system_prompt="You are the fraud supervisor. Coordinate triage, investigation, and decision.")

@app.entrypoint
def handle_request(user_input: str) -> str:
    try:
        result = supervisor(user_input)
        return str(result)
    except Exception as e:
        return json.dumps({"error": str(e), "action": "Escalate to human fraud team"})
'''

with open('multi_agent_runtime.py', 'w') as f:
    f.write(multi_agent_code)

print("Created multi_agent_runtime.py with error handling")
print("   - 4 fraud tools")
print("   - 3 specialist agents (triage, investigation, decision)")
print("   - Supervisor agent as @app.entrypoint")
print("   - try/except in every wrapper tool + entrypoint")
print()
print("To test locally:")
print("  from multi_agent_runtime import handle_request")
print("  result = handle_request('Investigate TXN-001 for CUST-001')")
print("  print(result)")

# Section 2: AgentCore Gateway — APIs as MCP Tools

## What is AgentCore Gateway?

AgentCore Gateway converts your existing **APIs and Lambda functions** into
**MCP-compatible tools** that any agent can discover and use — no code changes
to the original API.

```
Your existing API          AgentCore Gateway         Your Strands Agent
+------------------+      +------------------+      +------------------+
| REST API         | ---> | Auto-generates   | ---> | Discovers tools  |
| Lambda function  |      | MCP tool specs   |      | via MCP protocol |
| OpenAPI spec     |      | Handles auth     |      | Calls them       |
+------------------+      +------------------+      +------------------+
```

**Why this matters**: In production, your fraud investigation tools won't be
simulated Python functions — they'll be real REST APIs backed by databases.
Gateway lets your agents use those APIs without writing custom tool wrappers.

## Setting Up a Gateway (Console Walkthrough)

1. Open **Amazon Bedrock AgentCore** console
2. Go to **Gateways** > **Create gateway**
3. Add a **target** (Lambda function or API Gateway REST API)
4. Gateway auto-generates MCP tool definitions from your API spec
5. Connect your agent to the Gateway endpoint

## Connecting an Agent to Gateway Tools

```python
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# Connect to your Gateway endpoint
async with streamablehttp_client(
    url="https://your-gateway-url.execute-api.us-east-1.amazonaws.com",
    headers={"Authorization": f"Bearer {token}"}
) as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()
        # Discover available tools
        tools = await session.list_tools()
        print(f"Available tools: {[t.name for t in tools.tools]}")
```

In [ ]:
# =============================================================================
# DEMO: Simulating Gateway Tool Discovery
# =============================================================================
# Since setting up a real Gateway requires console access, we'll simulate
# how an agent would discover and use tools from a Gateway endpoint.

# Simulate what Gateway returns as MCP tool definitions
gateway_tools = [
    {
        "name": "query_transaction_db",
        "description": "Query the fraud transaction database via REST API",
        "inputSchema": {
            "type": "object",
            "properties": {
                "transaction_id": {"type": "string", "description": "Transaction ID"},
                "include_history": {"type": "boolean", "description": "Include full history"},
            },
            "required": ["transaction_id"]
        }
    },
    {
        "name": "run_ofac_screening",
        "description": "Run OFAC sanctions screening via compliance API",
        "inputSchema": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "Name to screen"},
                "country": {"type": "string", "description": "Country of origin"},
            },
            "required": ["name", "country"]
        }
    },
]

print("Simulated Gateway Tool Discovery:")
print(f"  Gateway URL: https://abc123.execute-api.us-east-1.amazonaws.com/mcp")
print(f"  Tools discovered: {len(gateway_tools)}")
for t in gateway_tools:
    print(f"    - {t['name']}: {t['description']}")
print()
print("In production, these tools are auto-generated from your REST APIs.")
print("Your agent calls them via MCP protocol — no custom wrapper code needed.")

## Lab 2: Build a Simulated Gateway Integration (15 minutes)

### Your Task

Create a "gateway simulator" — a function that mimics how AgentCore Gateway
converts a REST API response into a tool result that an agent can consume.

### Steps

1. **Create a `simulate_gateway_call` function** that:
   - Takes a tool name and parameters
   - Simulates calling a REST API (use a dict of mock API responses)
   - Returns the response in the MCP tool result format
2. **Wrap it as a Strands `@tool`** so your agent can call it
3. **Create an agent** that uses your gateway-simulated tool alongside the
   regular fraud tools. Test it on TXN-001.

### Expected Output

- A working gateway simulator that handles `query_transaction_db` and `run_ofac_screening`
- An agent that seamlessly mixes local tools and "gateway" tools

In [ ]:
# =============================================================================
# SOLUTION: LAB 2 — SIMULATED GATEWAY INTEGRATION
# =============================================================================

# Step 1: Create the gateway simulator tools
@tool
def gateway_query_transaction_db(transaction_id: str, include_history: bool = False) -> str:
    """Query the fraud transaction database via simulated Gateway REST API.

    Args:
        transaction_id: Transaction ID to query
        include_history: Whether to include full transaction history
    """
    # Simulates what a real REST API would return
    db = {
        "TXN-001": {
            "transaction_id": "TXN-001", "amount": 4500, "currency": "USD",
            "type": "wire_transfer", "merchant": "Unknown Overseas",
            "timestamp": "2026-03-15T03:47:00Z", "status": "pending_review",
            "origin_country": "US", "destination_country": "Nigeria",
        },
    }
    txn = db.get(transaction_id)
    if not txn:
        return json.dumps({"error": f"{transaction_id} not found", "status": 404})

    result = {"status": 200, "data": txn}
    if include_history:
        result["history"] = [
            {"date": "2026-03-10", "amount": 150, "merchant": "Grocery Store"},
            {"date": "2026-03-08", "amount": 89.99, "merchant": "Netflix"},
        ]
    return json.dumps(result, indent=2)

@tool
def gateway_run_ofac_screening(name: str, country: str) -> str:
    """Run OFAC sanctions screening via simulated Gateway compliance API.

    Args:
        name: Name to screen against OFAC list
        country: Country of origin
    """
    # Simulated OFAC screening
    high_risk_countries = ["Iran", "North Korea", "Syria", "Cuba"]
    monitored_countries = ["Nigeria", "Russia", "Venezuela"]

    if country in high_risk_countries:
        screening = "BLOCKED"
        detail = f"{country} is under comprehensive OFAC sanctions"
    elif country in monitored_countries:
        screening = "FLAGGED"
        detail = f"{country} is OFAC-monitored — enhanced due diligence required"
    else:
        screening = "CLEAR"
        detail = f"No OFAC restrictions for {country}"

    return json.dumps({
        "status": 200,
        "screening_result": screening,
        "name_screened": name,
        "country": country,
        "detail": detail,
        "timestamp": "2026-03-27T14:30:00Z",
    }, indent=2)


# Step 2: Create agent with mixed local + gateway tools
gateway_agent = Agent(
    model=llm,
    tools=[rt_lookup_transaction, rt_calculate_risk, gateway_query_transaction_db,
           gateway_run_ofac_screening],
    system_prompt=(
        "You are a fraud investigator with access to both local tools and "
        "gateway-connected APIs. Use gateway_query_transaction_db for detailed "
        "database queries and gateway_run_ofac_screening for sanctions checks. "
        "Use local tools for risk calculation. Provide a comprehensive report."
    ),
)

# Step 3: Test
print("Testing agent with mixed local + gateway tools...")
print("=" * 60)
gateway_result = gateway_agent(
    "Investigate TXN-001. Query the full database record with history, "
    "run OFAC screening for Nigeria, and calculate the risk score."
)
print("=" * 60)
print(f"\n{gateway_result}")
print("\n✅ Lab 2 complete!")

# Section 3: Human-in-the-Loop Patterns

## When Agents Need Human Approval

Not every decision should be automated. For HIGH risk fraud cases, you may want
a human to review and approve before the agent takes action (e.g., blocking an
account or filing a regulatory report).

**Pattern**: Add an "approval gate" tool that pauses execution and waits for
human input before continuing.

```python
@tool
def request_human_approval(case_summary: str, proposed_action: str) -> str:
    """Request human approval for a high-risk fraud decision.

    Args:
        case_summary: Summary of the investigation findings
        proposed_action: The action the agent wants to take
    """
    print(f"\n{'='*60}")
    print(f"HUMAN APPROVAL REQUIRED")
    print(f"{'='*60}")
    print(f"Case: {case_summary}")
    print(f"Proposed action: {proposed_action}")
    print(f"{'='*60}")
    decision = input("Approve? (yes/no/modify): ").strip().lower()
    if decision == "yes":
        return json.dumps({"approved": True, "action": proposed_action})
    elif decision == "modify":
        new_action = input("Enter modified action: ")
        return json.dumps({"approved": True, "action": new_action})
    else:
        return json.dumps({"approved": False, "reason": "Human reviewer rejected"})
```

This is a simple synchronous pattern. In production with AgentCore Runtime, you
would use async webhooks or SNS notifications instead of `input()`.

## Lab 3: Add Human-in-the-Loop to Your Pipeline (15 minutes)

### Your Task

Add a human approval gate to the multi-agent fraud pipeline so that HIGH-risk
decisions require human sign-off before execution.

### Steps

1. **Create a `request_human_approval` tool** using the pattern shown above
2. **Update the supervisor's system prompt** to include:
   "If the triage result is HIGH risk, call `request_human_approval` before
   calling `run_decision`. Include the investigation findings in the case summary."
3. **Test with TXN-001** (HIGH risk) — the agent should pause for your approval
4. **Test with TXN-002** (LOW risk) — the agent should skip approval entirely

### Expected Output

- HIGH risk transactions pause for human approval
- LOW risk transactions flow through without interruption
- The final verdict reflects whether human approval was granted or denied

In [ ]:
# =============================================================================
# SOLUTION: LAB 3 — HUMAN-IN-THE-LOOP PIPELINE
# =============================================================================

# Step 1: Create the approval gate tool
@tool
def request_human_approval(case_summary: str, proposed_action: str) -> str:
    """Request human approval for a high-risk fraud decision.

    Args:
        case_summary: Summary of the investigation findings
        proposed_action: The action the agent wants to take
    """
    print(f"\n{'='*60}")
    print(f"HUMAN APPROVAL REQUIRED")
    print(f"{'='*60}")
    print(f"Case Summary: {case_summary[:200]}...")
    print(f"Proposed Action: {proposed_action}")
    print(f"{'='*60}")

    decision = input("Approve? (yes/no/modify): ").strip().lower()

    if decision == "yes":
        return json.dumps({
            "approved": True,
            "action": proposed_action,
            "reviewer_note": "Approved by human reviewer"
        }, indent=2)
    elif decision == "modify":
        new_action = input("Enter modified action: ")
        return json.dumps({
            "approved": True,
            "action": new_action,
            "reviewer_note": "Modified by human reviewer"
        }, indent=2)
    else:
        reason = input("Reason for rejection (optional): ") or "No reason given"
        return json.dumps({
            "approved": False,
            "reason": reason,
            "reviewer_note": "Rejected by human reviewer"
        }, indent=2)


# Step 2: Create supervisor with approval gate
hitl_supervisor = Agent(
    model=llm,
    tools=[run_triage, run_investigation, request_human_approval, run_decision],
    system_prompt=(
        "You are the fraud operations supervisor with human oversight. "
        "Follow this pipeline: "
        "1) ALWAYS run_triage first "
        "2) If risk is MEDIUM or HIGH, run_investigation "
        "3) If risk is HIGH, you MUST call request_human_approval before "
        "   making a final decision. Include the investigation findings "
        "   in the case_summary and your proposed action. "
        "4) If risk is LOW or MEDIUM, skip approval and go to run_decision "
        "5) ALWAYS end with run_decision, incorporating approval status if applicable "
        "Report clearly whether human approval was obtained."
    ),
)

# Step 3: Test with HIGH risk (should ask for approval)
print("Testing HIGH risk transaction (will pause for your approval)...")
print("=" * 60)
high_risk_result = hitl_supervisor(
    "Investigate TXN-001 for CUST-001 — $4,500 wire transfer to Nigeria at 3:47 AM."
)
print("=" * 60)
print(f"\n{high_risk_result}")
print("\n✅ Lab 3 complete!")

# Summary

## What You Explored

1. **AgentCore Runtime** — Package agents as Python apps, deploy to managed containers
2. **AgentCore Gateway** — Convert REST APIs into MCP tools automatically
3. **Human-in-the-Loop** — Add approval gates for high-risk decisions

## Key Takeaways

- AgentCore Runtime uses `@app.entrypoint` to expose your agent as a service
- Gateway eliminates custom tool wrappers — your existing APIs become agent tools
- HITL patterns are essential for regulated industries like financial services
- In production, use async notifications (SNS/EventBridge) instead of `input()`

## Further Reading

- [AgentCore Documentation](https://docs.aws.amazon.com/bedrock-agentcore/)
- [Strands Agents SDK](https://strandsagents.com/)
- [AgentCore Samples on GitHub](https://github.com/awslabs/amazon-bedrock-agentcore-samples)